In [1]:
import json
import pandas as pd
from pathlib import Path

In [2]:
input_path = Path("../data/risk_dataset_v3.csv")

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (270, 7)


,license,license_family,project_type,distribution_model,usage,risk,reason
0,EPL-2.0,Weak Copyleft,robotics control system,distributed,frontend component,Medium,File-level copyleft obligations create moderat...
1,MIT,Permissive,banking middleware,hosted,plugin framework,Low,Low risk because the license allows proprietar...
2,MIT,Permissive,AI inference platform,hosted,utility package,Low,License creates minimal compliance burden for ...
3,Proprietary,Restricted,robotics control system,internal-only,workflow engine,High,Strong copyleft obligations may require source...
4,GPL-3.0,Strong Copyleft,cloud CRM platform,hosted,data processing module,High,High risk because network or redistribution ob...


In [3]:
required_columns = [
    "license",
    "license_family",
    "project_type",
    "distribution_model",
    "usage",
    "risk",
    "reason"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [4]:
def build_user_prompt(row):
    return f"""
You are an open-source license compliance assistant.

Classify the compliance risk for the following software dependency scenario.

License: {row["license"]}
License Family: {row["license_family"]}
Project Type: {row["project_type"]}
Distribution Model: {row["distribution_model"]}
Usage: {row["usage"]}

Return exactly:
Risk: Low/Medium/High
Reason: one short sentence
""".strip()


def build_assistant_response(row):
    return f"""Risk: {row["risk"]}
Reason: {row["reason"]}"""

In [5]:
records = []

for _, row in df.iterrows():
    sample = {
        "messages": [
            {
                "role": "user",
                "content": build_user_prompt(row)
            },
            {
                "role": "assistant",
                "content": build_assistant_response(row)
            }
        ]
    }

    records.append(sample)

print("Total fine-tuning samples:", len(records))
records[0]

Total fine-tuning samples: 270


{'messages': [{'role': 'user',
   'content': 'You are an open-source license compliance assistant.\n\nClassify the compliance risk for the following software dependency scenario.\n\nLicense: EPL-2.0\nLicense Family: Weak Copyleft\nProject Type: robotics control system\nDistribution Model: distributed\nUsage: frontend component\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'},
  {'role': 'assistant',
   'content': 'Risk: Medium\nReason: File-level copyleft obligations create moderate compliance requirements'}]}

In [6]:
output_path = Path("../data/qwen_finetune_dataset_v2.jsonl")

with open(output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved fine-tuning dataset to: {output_path}")

Saved fine-tuning dataset to: ../data/qwen_finetune_dataset_v2.jsonl


In [7]:
validation_records = []

with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        validation_records.append(json.loads(line))

print("Loaded records:", len(validation_records))
print(validation_records[0])

Loaded records: 270
{'messages': [{'role': 'user', 'content': 'You are an open-source license compliance assistant.\n\nClassify the compliance risk for the following software dependency scenario.\n\nLicense: EPL-2.0\nLicense Family: Weak Copyleft\nProject Type: robotics control system\nDistribution Model: distributed\nUsage: frontend component\n\nReturn exactly:\nRisk: Low/Medium/High\nReason: one short sentence'}, {'role': 'assistant', 'content': 'Risk: Medium\nReason: File-level copyleft obligations create moderate compliance requirements'}]}


In [8]:
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 216
Validation samples: 54


In [9]:
def dataframe_to_jsonl(dataframe, path):
    records = []

    for _, row in dataframe.iterrows():
        sample = {
            "messages": [
                {
                    "role": "user",
                    "content": build_user_prompt(row)
                },
                {
                    "role": "assistant",
                    "content": build_assistant_response(row)
                }
            ]
        }

        records.append(sample)

    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} samples to {path}")

In [10]:
dataframe_to_jsonl(train_df, "../data/qwen_finetune_train_v2.jsonl")
dataframe_to_jsonl(val_df, "../data/qwen_finetune_val_v2.jsonl")

Saved 216 samples to ../data/qwen_finetune_train_v2.jsonl
Saved 54 samples to ../data/qwen_finetune_val_v2.jsonl
